# 01 – Data loading and sanity checks

AI4I 2020 Predictive Maintenance Dataset (UCI). This notebook verifies the integrity of the raw data, checks the consistency of the failure labels and writes the cleaned dataset used by all later notebooks.

In [ ]:
import numpy as np
import pandas as pd

from config import RAW_DATA, CLEAN_DATA, RESULTS_DIR, TARGET, FAILURE_MODES

pd.set_option("display.max_columns", None)

## 1. Load the raw data

The CSV starts with a UTF-8 byte-order mark, so it is read with `utf-8-sig` to keep the first column name clean.

In [ ]:
df = pd.read_csv(RAW_DATA, encoding="utf-8-sig")
print(df.shape)
df.head()

In [ ]:
df.dtypes

In [ ]:
df.describe().T

## 2. Integrity checks

In [ ]:
integrity = pd.Series({
    "Missing values": int(df.isna().sum().sum()),
    "Duplicate rows": int(df.duplicated().sum()),
    "Duplicate UDI": int(df["UDI"].duplicated().sum()),
    "Duplicate Product ID": int(df["Product ID"].duplicated().sum()),
})
integrity

## 3. Product type and class balance

In [ ]:
type_counts = df["Type"].value_counts()
pd.DataFrame({"count": type_counts, "share": (type_counts / len(df)).round(4)})

In [ ]:
target_counts = df[TARGET].value_counts()
pd.DataFrame({"count": target_counts, "share": (target_counts / len(df)).round(4)})

## 4. Failure modes

TWF: tool wear failure, HDF: heat dissipation failure, PWF: power failure, OSF: overstrain failure, RNF: random failure. A failed machine can show more than one mode.

In [ ]:
mode_count = df[FAILURE_MODES].sum(axis=1)
df[FAILURE_MODES].sum().rename("count").to_frame()

In [ ]:
mode_count[df[TARGET] == 1].value_counts().sort_index().rename_axis("modes per failure").rename("count").to_frame()

## 5. Label consistency

According to the dataset documentation, `Machine failure` is 1 if and only if at least one failure mode is 1. Rows that break this rule are identified below.

In [ ]:
failure_without_mode = (df[TARGET] == 1) & (mode_count == 0)
mode_without_failure = (df[TARGET] == 0) & (mode_count > 0)

pd.crosstab(df[TARGET], (mode_count > 0).rename("Any failure mode"))

In [ ]:
df.loc[mode_without_failure, FAILURE_MODES].sum().rename("count").to_frame()

The failures without a recorded mode are compared with the documented failure conditions (HDF, PWF and OSF thresholds from the UCI description; for TWF the tool-wear range observed for TWF failures in this dataset).

In [ ]:
speed = df["Rotational speed [rpm]"]
temp_diff = df["Process temperature [K]"] - df["Air temperature [K]"]
power = df["Torque [Nm]"] * speed * 2 * np.pi / 60
strain = df["Tool wear [min]"] * df["Torque [Nm]"]
strain_limit = df["Type"].map({"L": 11000, "M": 12000, "H": 13000})

rules = pd.DataFrame({
    "HDF": (temp_diff < 8.6) & (speed < 1380),
    "PWF": (power < 3500) | (power > 9000),
    "OSF": strain > strain_limit,
})
agreement = {
    mode: pd.crosstab(df[mode], rules[mode]).reindex(index=[0, 1], columns=[False, True], fill_value=0).to_numpy().ravel()
    for mode in rules
}
pd.DataFrame(agreement, index=["flag 0 / rule 0", "flag 0 / rule 1", "flag 1 / rule 0", "flag 1 / rule 1"])

In [ ]:
twf_wear = df.loc[df["TWF"] == 1, "Tool wear [min]"]
print(f"Tool wear range of TWF failures: {twf_wear.min()}–{twf_wear.max()} min")

rules["TWF window"] = df["Tool wear [min]"].between(twf_wear.min(), twf_wear.max())
df.loc[failure_without_mode, ["UDI", "Type"]].join(rules)

The documented rules reproduce the HDF, PWF and OSF flags without a single mismatch. None of the failures without a mode satisfies any of these rules; one of them lies inside the TWF wear window, but TWF is random within that window and was not flagged. Together with the RNF rows labelled as non-failures, these rows contradict the labelling rule and are treated as label noise.

## 6. Remove inconsistent rows and save the cleaned dataset

In [ ]:
inconsistent = failure_without_mode | mode_without_failure
clean = df.loc[~inconsistent].reset_index(drop=True)

print(f"Removed rows: {inconsistent.sum()} ({failure_without_mode.sum()} failures without mode, "
      f"{mode_without_failure.sum()} modes without failure)")
print(f"Clean dataset: {clean.shape}, failures: {clean[TARGET].sum()} ({clean[TARGET].mean():.2%})")

clean.to_csv(CLEAN_DATA, index=False)

In [ ]:
summary = pd.Series({
    "Raw rows": len(df),
    "Raw failures": int(df[TARGET].sum()),
    "Failures without mode (removed)": int(failure_without_mode.sum()),
    "Modes without failure (removed)": int(mode_without_failure.sum()),
    "Clean rows": len(clean),
    "Clean failures": int(clean[TARGET].sum()),
    "Clean failure rate": round(clean[TARGET].mean(), 4),
}, name="value", dtype=object)
summary.to_csv(RESULTS_DIR / "01_data_check_summary.csv")
summary